In [1]:
import os

import rasterio
from rasterio.mask import mask
from shapely.geometry import Polygon, mapping, box
import geopandas as gpd
import numpy as np
import folium
from rasterio.mask import mask
import shutil
import glob
import pickle
import warnings
warnings.filterwarnings("ignore")
import matplotlib
%matplotlib widget
import matplotlib.pyplot as plt
plt.ion()
from datetime import datetime
from Toolshed import Download, Toolbox, VegetationLine, Plotting, PlottingSeaborn, Transects
import ee
import geopandas as gpd
import geemap
from shapely.geometry import MultiPolygon
matplotlib.rcParams['font.family'] = 'DejaVu Sans'

ee.Initialize()
ee.Authenticate() # should only need to be run the first time after installation

True

In [2]:
# The points represent the corners of a bounding box that go around your site
sitename = 'Riverbank_South'
# ConnorsCove_BlackBeach  Gooseberry_Cove  Rivermouth_West
# Marshland, Riverbank_North, Riverbank_South, WestBranch_Reservoir

# Date range
# dates = ['2023-04-01', '2023-10-30', '2024-04-01', '2024-10-30', '2025-04-01', '2025-10-30']
dates = ['2020-10-01', '2025-06-01']

# Satellite missions
# Input a list of containing any/all of 'L5', 'L7', 'L8', 'L9', 'S2', 'PSScene4Band'
# L5: 1984-2013; L7: 1999-2017 (SLC error from 2003); L8: 2013-present; S2: 2014-present; L9: 2021-present
sat_list = ['PSScene4Band']

# Cloud threshold for screening out cloudy imagery (0.5 or 50% recommended)
cloud_thresh = 0.3

# Extract shoreline (wet-dry boundary) as well as veg edge
wetdry = True

# Directory where the data will be stored
filepath = Toolbox.CreateFileStructure(sitename, sat_list)

In [3]:
if len(dates)>2:
    daterange='no'
else:
    daterange='yes'
years = list(Toolbox.daterange(datetime.strptime(dates[0],'%Y-%m-%d'), datetime.strptime(dates[-1],'%Y-%m-%d')))

In [4]:
import os
import shutil
from datetime import datetime

# define valid date range(s)
valid_months = list(range(1, 13))  # April (4) to October (11)
valid_years = [2020, 2021, 2022, 2023, 2024, 2025]

# Define source folders
source_folder_images = os.path.join(filepath, 'PlanetScope')           # Images
source_folder_cloudmasks = os.path.join(filepath, 'PlanetScope', 'cloudmasks')  # Cloud masks

# Define target folders
target_image_folder = os.path.join(filepath, sitename, 'local_images', 'PSScene4Band')
target_cloudmask_folder = os.path.join(target_image_folder, 'cloudmasks')

# Create target folders if they don't exist
os.makedirs(target_image_folder, exist_ok=True)
os.makedirs(target_cloudmask_folder, exist_ok=True)

def is_valid_date(filename):
    try:
        # Step 2: extract date string from filename like '20230622_150103_composite.tif'
        date_str = filename.split('_')[0]  # '20230622'
        date_obj = datetime.strptime(date_str, '%Y%m%d')
        return date_obj.year in valid_years and date_obj.month in valid_months
    except Exception:
        return False

# Copy images
for file in os.listdir(source_folder_images):
    if file.endswith('_composite.tif') and is_valid_date(file):
        full_file_path = os.path.join(source_folder_images, file)
        shutil.copy2(full_file_path, os.path.join(target_image_folder, file))
        print(f"Copied image: {file}")

# Copy cloud masks
for file in os.listdir(source_folder_cloudmasks):
    if file.endswith('_composite_udm2.tif') and is_valid_date(file):
        full_file_path = os.path.join(source_folder_cloudmasks, file)
        shutil.copy2(full_file_path, os.path.join(target_cloudmask_folder, file))
        print(f"Copied cloud mask: {file}")

print("done")

Copied image: 20210729_151631_composite.tif
Copied image: 20210914_144042_composite.tif
Copied image: 20211221_144107_composite.tif
Copied image: 20210801_144436_composite.tif
Copied image: 20201012_143405_composite.tif
Copied image: 20210825_143103_composite.tif
Copied image: 20210907_143014_composite.tif
Copied image: 20211205_151919_composite.tif
Copied image: 20211102_143030_composite.tif
Copied image: 20210920_151237_composite.tif
Copied image: 20211011_142838_composite.tif
Copied image: 20210826_143142_composite.tif
Copied image: 20210817_151931_composite.tif
Copied image: 20210831_144339_composite.tif
Copied image: 20211220_142437_composite.tif
Copied image: 20210815_152019_composite.tif
Copied image: 20201122_143553_composite.tif
Copied image: 20211106_142747_composite.tif
Copied image: 20210919_144332_composite.tif
Copied image: 20211212_143704_composite.tif
Copied image: 20211005_151326_composite.tif
Copied image: 20210827_143946_composite.tif
Copied image: 20201119_143635_co

In [5]:
# --- Define folders ---
base_folder = os.path.join("Data", sitename, "local_images", "PSScene4Band")
cloudmask_folder = os.path.join(base_folder, "cloudmasks")

# --- Store updated georef info for both folders ---
updated_georef = {}

In [6]:
def crop_and_overwrite(path, filename, folder_label, sitename):
    # Load AOI polygon and GeoDataFrame from file
    polygon, point, gdf, lonmin, lonmax, latmin, latmax = Toolbox.AOI_from_file(sitename)
    polygon_geom = gdf.geometry.values[0]

    with rasterio.open(path, 'r') as src:
        print(f"\n{filename}")

        # Reproject polygon to match raster CRS
        gdf_proj = gdf.to_crs(src.crs)

        # DEBUG: Print bounds
        print("Polygon bounds (reprojected):", gdf_proj.bounds.values[0])
        print("Raster bounds:", src.bounds)

        # Get raster bounds as geometry
        raster_geom = gpd.GeoSeries([box(*src.bounds)], crs=src.crs)

        # Use intersection or force clipped overlap
        poly_geom = gdf_proj.geometry[0]

        if not poly_geom.intersects(raster_geom[0]):
            print(f" Forcing bbox clip for {filename}")
            clipped_poly = poly_geom.intersection(raster_geom[0].buffer(1))  # slight buffer
        else:
            clipped_poly = poly_geom.intersection(raster_geom[0])

        if clipped_poly.is_empty:
            print(f" Even forced clip failed. Skipped {filename}")
            return

        # Make GeoDataFrame from clipped polygon
        gdf_intersect = gpd.GeoDataFrame({'geometry': [clipped_poly]}, crs=src.crs)

        # Crop image
        try:
            out_image, out_transform = mask(src, gdf_intersect.geometry.map(mapping), crop=True)
        except Exception as e:
            print(f"Error cropping image {filename}: {e}")
            return

        out_meta = src.meta.copy()
        out_meta.update({
            "height": out_image.shape[1],
            "width": out_image.shape[2],
            "transform": out_transform
        })

    # Overwrite the raster
    with rasterio.open(path, "w", **out_meta) as dst:
        dst.write(out_image)

    # Optional: save updated AOI for logging or chaining
    updated_georef[f"{folder_label}/{filename}"] = {
        "transform": out_transform,
        "shape": (out_image.shape[1], out_image.shape[2]),
        "crs": src.crs.to_string()
    }

    print(f" Cropped and saved: {folder_label}/{filename}")

In [7]:
# --- Process main folder ---
print("Cropping composite images...")
for filename in os.listdir(base_folder):
    if filename.endswith(".tif") and filename != "cloudmasks":
        crop_and_overwrite(
            path=os.path.join(base_folder, filename),
            filename=filename,
            folder_label="composites",
            sitename=sitename
        )

# --- Process cloudmasks folder ---
print("\nCropping cloudmask images...")
for filename in os.listdir(cloudmask_folder):
    if filename.endswith(".tif"):
        crop_and_overwrite(
            path=os.path.join(cloudmask_folder, filename),
            filename=filename,
            folder_label="cloudmasks",
            sitename=sitename
        )

# --- Print update instructions ---
print("\nTo update metadata['acc_georef']:\n")
for fn, val in updated_georef.items():
    print(f"# {fn}")
    print(f"metadata['acc_georef'][i]['transform'] = {repr(val['transform'])}")
    print(f"metadata['acc_georef'][i]['shape'] = {val['shape']}")
    print(f"metadata['epsg'] = '{val['crs']}'\n")

Cropping composite images...

20210729_151631_composite.tif
Polygon bounds (reprojected): [ 710578.7429073  5004868.77937798  715172.57267512 5008077.37452535]
Raster bounds: BoundingBox(left=705942.0, bottom=5001375.0, right=719736.0, top=5009361.0)
 Cropped and saved: composites/20210729_151631_composite.tif

20210914_144042_composite.tif
Polygon bounds (reprojected): [ 710578.7429073  5004868.77937798  715172.57267512 5008077.37452535]
Raster bounds: BoundingBox(left=705942.0, bottom=5001375.0, right=719736.0, top=5009361.0)
 Cropped and saved: composites/20210914_144042_composite.tif

20211221_144107_composite.tif
Polygon bounds (reprojected): [ 710578.7429073  5004868.77937798  715172.57267512 5008077.37452535]
Raster bounds: BoundingBox(left=705942.0, bottom=5001375.0, right=719736.0, top=5009361.0)
 Cropped and saved: composites/20211221_144107_composite.tif

20210801_144436_composite.tif
Polygon bounds (reprojected): [ 710578.7429073  5004868.77937798  715172.57267512 5008077.3

In [8]:
'''
if not os.path.exists(referenceLinePath):
    print(f" Reference line shapefile not found: {referenceLinePath}")
else:
    # Read the reference line shapefile
    referenceLineDF = gpd.read_file(referenceLinePath)

    # Reproject reference lines to match gdf's CRS
    referenceLineDF = referenceLineDF.to_crs(gdf.crs)

    # Clip reference lines to updated gdf polygon (already cropped)
    clipped_ref = gpd.overlay(referenceLineDF, gdf, how='intersection')

    if clipped_ref.empty:
        print(" Clipping removed all reference lines. Nothing to save.")
    else:
        # Overwrite the shapefile with clipped reference lines
        clipped_ref.to_file(referenceLinePath)
        print(f" Clipped and saved: {referenceLinePath}")'''

'\nif not os.path.exists(referenceLinePath):\n    print(f" Reference line shapefile not found: {referenceLinePath}")\nelse:\n    # Read the reference line shapefile\n    referenceLineDF = gpd.read_file(referenceLinePath)\n\n    # Reproject reference lines to match gdf\'s CRS\n    referenceLineDF = referenceLineDF.to_crs(gdf.crs)\n\n    # Clip reference lines to updated gdf polygon (already cropped)\n    clipped_ref = gpd.overlay(referenceLineDF, gdf, how=\'intersection\')\n\n    if clipped_ref.empty:\n        print(" Clipping removed all reference lines. Nothing to save.")\n    else:\n        # Overwrite the shapefile with clipped reference lines\n        clipped_ref.to_file(referenceLinePath)\n        print(f" Clipped and saved: {referenceLinePath}")'